# Adaptive Market Foundation — Google Colab T4
Training pipeline: data validation → market tokenizer → GPT pretraining → multi-asset fine-tuning → evaluation artifacts.

Set Colab Runtime to **T4 GPU** before running. 2023 is reserved for final testing; do not tune on it.

In [ ]:
!nvidia-smi
!git clone --branch adaptive-colab-t4-v2 https://github.com/NodePro-gif/market-foundation-model.git /content/market-foundation-model
%cd /content/market-foundation-model
!pip -q install -r requirements-colab.txt


In [ ]:
# Static syntax check before starting expensive GPU jobs.
!python -m py_compile stage1_corpus.py stage2_tokenizer.py stage3_pretrain.py stage4_finetune.py train_multisymbol.py backtest_multisymbol.py market_intelligence.py
print('Python syntax checks passed')
!python backtest_multisymbol.py
shutil.copytree('/content/adaptive_results', DRIVE_ROOT+'/adaptive_results', dirs_exist_ok=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
DRIVE_ROOT='/content/drive/MyDrive/market-foundation-model'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs('/content/mkt', exist_ok=True)
if os.path.isdir(DRIVE_ROOT+'/mkt'):
    shutil.copytree(DRIVE_ROOT+'/mkt','/content/mkt',dirs_exist_ok=True)
    print('Loaded tokenized data from Drive')
else:
    print('No cached /mkt on Drive; Stage 1 + Stage 2 will run')


In [ ]:
# Run this cell only when /content/mkt is empty or tokenizer/corpus needs rebuilding.
if not os.path.exists('/content/mkt/vocab.json'):
    !python stage1_corpus.py
    !python stage2_tokenizer.py
    shutil.copytree('/content/mkt', DRIVE_ROOT+'/mkt', dirs_exist_ok=True)
else:
    print('Using existing tokenized corpus; skipping expensive rebuild.')
!python test_tokenizer.py


In [ ]:
# Full foundation pretraining. 2022 is validation; 2023 is untouched final test.
import os
os.environ['PRETRAIN_BATCH']='48'       # lower to 32 or 16 if CUDA OOM
os.environ['PRETRAIN_MAX_STEPS']='6000'
os.environ['PRETRAIN_CKPT_EVERY']='250'
os.environ['PRETRAIN_EVAL_EVERY']='250'
os.environ['PRETRAIN_LR']='0.0003'
!python stage3_pretrain.py
shutil.copytree('/content/ckpt', DRIVE_ROOT+'/ckpt', dirs_exist_ok=True)


In [ ]:
# Validate that checkpoint embeddings match the current BPE tokenizer.
os.environ['CKPT']='/content/ckpt/best.pt'
!python verify_ckpt.py


In [ ]:
# Multi-asset supervised fine-tuning. NIKKEI/BRENT/EURGBP remain held out.
os.environ['CKPT']='/content/ckpt/best.pt'
os.environ['ADAPT_BATCH']='48'
os.environ['ADAPT_STRIDE']='8'
os.environ['ADAPT_EPOCHS']='8'
os.environ['ADAPT_LR']='0.00008'
!python train_multisymbol.py
shutil.copytree('/content/adaptive_results', DRIVE_ROOT+'/adaptive_results', dirs_exist_ok=True)


## After training
Inspect `adaptive_results/metrics.json`. Compare validation and final 2023 test metrics, then run chronological walk-forward tests with realistic spread, commission and slippage. The held-out symbols are transfer diagnostics only. If acceptance gates fail, keep the model in research-only mode. No live trading is enabled.